### 실습 1. 개선 데이터 확인하기

먼저 모델을 바꾸기 전에 데이터를 확인합니다.

In [ ]:
# CSV 데이터를 불러오기 위해 pandas를 불러온다.
import pandas as pd

# 분석에 사용할 통합 도서 데이터를 불러온다.
df = pd.read_csv(
    "books_improved.csv",
    encoding="utf-8-sig"
)

# 전체 행과 열의 개수를 확인한다.
print("데이터 크기:", df.shape)

# 데이터의 컬럼명을 확인한다.
print("컬럼명:", df.columns.tolist())

# 데이터의 앞부분 5행을 확인한다.
display(df.head())

데이터 크기: (5672, 12)
컬럼명: ['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물', '출판사', '발행(출시)일자', '분야']


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설
3,9791175772892,S000220307771,테오,"21,000","18,900",10%,5%,"1,050",앨런 레비,오팬하우스,20260701,소설
4,9791198547514,S000211748790,수족관,"17,700","15,930",10%,5%,880,유래혁,포스터샵,20240111,소설


In [3]:
# 분야별 도서 개수를 확인한다.
category_counts = df["분야"].value_counts(dropna=False)

print("분야별 도서 수")
print(category_counts)

# 전체 카테고리 개수와 분야 결측치를 확인한다.
print("\n카테고리 수:", df["분야"].nunique(dropna=False))
print("분야 결측치:", df["분야"].isna().sum())

분야별 도서 수
분야
소설           1000
시/에세이        1000
인문           1000
경제/경영         999
컴퓨터/IT        645
취미/실용/스포츠     357
건강            267
가정/육아         216
요리            188
Name: count, dtype: int64

카테고리 수: 9
분야 결측치: 0


In [4]:
# 각 컬럼에 비어 있는 값이 몇 개인지 확인한다.
print("컬럼별 결측치")
print(df.isna().sum())

# 판매상품 ID와 상품명 중복 개수를 확인한다.
print("\n중복 판매상품 ID:", df["판매상품 ID"].duplicated().sum())
print("중복 상품명:", df["상품명"].duplicated().sum())

컬럼별 결측치
상품코드        0
판매상품 ID     0
상품명         0
정가          0
판매가         0
할인율         0
적립율         0
적립예정포인트     0
인물          0
출판사         0
발행(출시)일자    0
분야          0
dtype: int64

중복 판매상품 ID: 0
중복 상품명: 71


In [5]:
# 같은 상품명이 두 번 이상 등장한 행을 모두 찾는다.
duplicate_titles = df[
    df["상품명"].duplicated(keep=False)
].sort_values(
    by=["상품명", "판매상품 ID"]
)

# 중복 상품명에 해당하는 전체 행 수를 확인한다.
print("중복 상품명 관련 전체 행 수:", len(duplicate_titles))

# 상품 ID, 제목, 인물, 출판사, 분야를 비교한다.
display(
    duplicate_titles[
        ["판매상품 ID", "상품명", "인물", "출판사", "분야"]
    ].head(20)
)

중복 상품명 관련 전체 행 수: 126


,판매상품 ID,상품명,인물,출판사,분야
29,S000000620214,1984,조지 오웰,민음사,소설
560,S000000777564,1984,조지 오웰,문학동네,소설
4689,S000219302825,IFRS 고급회계,신현걸 외,지승,경제/경영
4598,S000219971949,IFRS 고급회계,김기동,샘앤북스,경제/경영
4960,S000211998394,IFRS 회계원리,강정연 외,시대가치,경제/경영
4895,S000214180711,IFRS 회계원리,두서영 외,세학사,경제/경영
4151,S000218858292,IFRS 회계원리,김기동 외,샘앤북스,경제/경영
4205,S000219085074,IFRS 회계원리,최관 외,신영사,경제/경영
4544,S000220663384,IFRS 회계원리,홍진호,에스더블유제이,경제/경영
4431,S000001681326,K-IFRS 회계원리,김순기 외,박영사,경제/경영


In [6]:
# 상품명별로 몇 개의 서로 다른 분야가 연결되어 있는지 계산한다.
title_category_counts = (
    df.groupby("상품명")["분야"]
    .nunique()
)

# 동일한 상품명이 두 개 이상의 분야에 들어간 경우만 찾는다.
conflicting_titles = title_category_counts[
    title_category_counts > 1
]

print("여러 분야에 포함된 상품명 수:", len(conflicting_titles))

# 분야가 충돌하는 상품명의 상세 데이터를 확인한다.
conflicting_rows = df[
    df["상품명"].isin(conflicting_titles.index)
].sort_values(
    by=["상품명", "분야"]
)

display(
    conflicting_rows[
        ["판매상품 ID", "상품명", "인물", "출판사", "분야"]
    ].head(20)
)

여러 분야에 포함된 상품명 수: 1


,판매상품 ID,상품명,인물,출판사,분야
54,S000000620231,차라투스트라는 이렇게 말했다,프리드리히 니체,민음사,소설
1143,S000220956633,차라투스트라는 이렇게 말했다,프리드리히 니체,문학동네,시/에세이


## 실습 1. 개선 데이터 확인 결과

### 확인 결과

- 전체 데이터 크기는 **5,672행, 12열**이다.
- 도서 분야는 총 **9개**이다.
- 모든 컬럼의 결측치는 **0개**이다.
- 중복된 판매상품 ID는 **0개**이므로 각 상품은 서로 다른 상품 ID를 가지고 있다.
- 중복 상품명은 **71개**이며, 중복 상품명과 관련된 전체 행은 **126개**이다.
- 중복 상품은 제목이 같더라도 저자, 출판사, 판매상품 ID가 다른 경우가 있어 바로 삭제하지 않았다.
- 동일한 상품명이 여러 분야에 포함된 경우는 **1개**이다.
- `차라투스트라는 이렇게 말했다`가 `소설`과 `시/에세이`에 각각 포함되어 있었다.

### 분야별 데이터 수

- 소설: 1,000권
- 시/에세이: 1,000권
- 인문: 1,000권
- 경제/경영: 999권
- 컴퓨터/IT: 645권
- 취미/실용/스포츠: 357권
- 건강: 267권
- 가정/육아: 216권
- 요리: 188권

### 해석

결측치는 없지만 분야별 도서 수에는 차이가 있다.  
소설, 시/에세이, 인문은 각각 1,000권이지만 요리는 188권으로 가장 적다.

이러한 데이터 불균형으로 인해 모델이 데이터가 많은 분야를 더 자주 예측할 가능성이 있다. 또한 같은 제목이 서로 다른 분야에 포함된 사례가 있어 제목만으로 분야를 분류할 때 혼란이 생길 수 있다.

현재 단계에서는 원본 데이터를 삭제하거나 변경하지 않고 데이터의 특징과 문제점을 확인했다.

### 실습 2. Baseline을 먼저 측정하기

개선 효과를 말하려면 먼저 개선 전 기준점(Baseline) 이 필요합니다.

이번 Baseline은 Chapter 04에서 사용한 구조를 유지합니다.

In [7]:
# 데이터를 훈련용과 테스트용으로 나누기 위해 불러온다.
from sklearn.model_selection import train_test_split

# 도서 제목을 모델의 입력 데이터로 사용한다.
X = df["상품명"]

# 도서 분야를 모델이 맞혀야 하는 정답으로 사용한다.
y = df["분야"]

# 데이터를 훈련용 80%, 테스트용 20%로 나눈다.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 분리된 데이터의 개수를 확인한다.
print("전체 데이터:", len(X))
print("훈련 데이터:", len(X_train))
print("테스트 데이터:", len(X_test))

전체 데이터: 5672
훈련 데이터: 4537
테스트 데이터: 1135


In [8]:
# 도서 제목을 TF-IDF 숫자 벡터로 변환하기 위해 불러온다.
from sklearn.feature_extraction.text import TfidfVectorizer

# 도서 분야를 예측할 Naive Bayes 모델을 불러온다.
from sklearn.naive_bayes import MultinomialNB

# 기본 설정의 TF-IDF 변환기를 만든다.
baseline_vectorizer = TfidfVectorizer()

# 훈련 데이터에서 단어를 학습하고 숫자 벡터로 변환한다.
X_train_vec = baseline_vectorizer.fit_transform(X_train)

# 훈련 데이터에서 배운 기준으로 테스트 데이터를 변환한다.
X_test_vec = baseline_vectorizer.transform(X_test)

# 기본 분류 모델을 만든다.
baseline_model = MultinomialNB()

# 훈련용 제목과 분야를 이용해 모델을 학습시킨다.
baseline_model.fit(X_train_vec, y_train)

# 테스트 데이터의 분야를 예측한다.
baseline_pred = baseline_model.predict(X_test_vec)

# 변환된 데이터의 크기를 확인한다.
print("훈련 TF-IDF 크기:", X_train_vec.shape)
print("테스트 TF-IDF 크기:", X_test_vec.shape)
print("예측 결과 수:", len(baseline_pred))

훈련 TF-IDF 크기: (4537, 7874)
테스트 TF-IDF 크기: (1135, 7874)
예측 결과 수: 1135


In [9]:
# 모델의 정확도와 분야별 평가 결과를 확인하기 위해 불러온다.
from sklearn.metrics import accuracy_score, classification_report

# 실제 분야와 예측 분야를 비교해 정확도를 계산한다.
baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

# 전체 정확도를 출력한다.
print("Baseline Accuracy:", baseline_accuracy)

# 분야별 precision, recall, F1-score를 출력한다.
print(
    classification_report(
        y_test,
        baseline_pred
    )
)

Baseline Accuracy: 0.5374449339207048
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       0.80      0.08      0.14        53
       경제/경영       0.58      0.62      0.60       200
          소설       0.40      0.78      0.53       200
       시/에세이       0.56      0.57      0.57       200
          요리       1.00      0.05      0.10        38
          인문       0.52      0.49      0.51       200
   취미/실용/스포츠       1.00      0.15      0.27        72
      컴퓨터/IT       0.80      0.78      0.79       129

    accuracy                           0.54      1135
   macro avg       0.63      0.39      0.39      1135
weighted avg       0.59      0.54      0.50      1135



c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

## 실습 2. Baseline 측정 결과

### 데이터 분리

- 전체 데이터: 5,672권
- 훈련 데이터: 4,537권
- 테스트 데이터: 1,135권
- TF-IDF 단어 특성 수: 7,874개

전체 데이터를 훈련용 80%, 테스트용 20%로 분리했다.  
`stratify=y`를 사용하여 분야별 데이터 비율이 유지되도록 했다.

### Baseline 평가 결과

- Accuracy: **0.5374**
- 정확도: **약 53.74%**
- Macro F1-score: **0.39**
- 정답 수: **610 / 1,135**

### 분야별 결과 해석

- `컴퓨터/IT`의 F1-score가 **0.79**로 가장 높았다.
- `경제/경영`, `소설`, `시/에세이`, `인문`은 비교적 많은 학습 데이터를 가지고 있어 다른 분야보다 성능이 높았다.
- `가정/육아`는 모델이 한 번도 예측하지 못해 precision, recall, F1-score가 모두 0으로 나타났다.
- `요리`, `건강`, `취미/실용/스포츠`는 precision은 높지만 recall이 낮았다.
- `소설`은 recall이 0.78로 높지만 precision은 0.40으로 낮아, 다른 분야의 책도 소설로 많이 예측한 것으로 보인다.

### 결론

기본 TF-IDF와 Naive Bayes를 사용한 Baseline 정확도는 약 53.74%였다.

분야별 데이터 수가 다르기 때문에 데이터가 많은 분야를 중심으로 예측하는 경향이 나타났다. 특히 데이터가 적은 분야는 제대로 예측하지 못하는 문제가 확인되었다.

이 결과를 이후 한국어 형태소 분석과 단어 필터링을 적용한 모델의 비교 기준으로 사용한다.

### 실습 3. 한국어 형태소 분석 적용하기

기본 TF-IDF는 제목 문자열을 그대로 사용합니다.
 
이번에는 한국어 제목에서 의미 있는 형태소를 추출해 TF-IDF 입력을 바꿔 봅니다.

In [10]:
# 한국어 문장을 형태소 단위로 분석하기 위해 Kiwi를 불러온다.
from kiwipiepy import Kiwi

# 한국어 형태소 분석기를 만든다.
kiwi = Kiwi()

# 형태소 분석을 확인할 예시 제목을 저장한다.
sample = "처음 배우는 파이썬 데이터 분석"

# 예시 제목을 형태소 단위로 나누어 단어와 품사를 출력한다.
for token in kiwi.tokenize(sample):
    print(token.form, token.tag)

처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG


In [11]:
# 상품명에서 분류에 사용할 주요 단어를 추출하는 함수를 만든다.
def extract_title_tokens(title):
    selected_tokens = []

    # 상품명을 형태소 단위로 분석한다.
    for token in kiwi.tokenize(str(title)):

        # 일반 명사, 고유 명사, 영문 단어만 선택한다.
        if token.tag in {"NNG", "NNP", "SL"}:
            selected_tokens.append(token.form)

    # 선택한 단어를 공백으로 연결해 하나의 문자열로 반환한다.
    return " ".join(selected_tokens)


# 모든 상품명에 형태소 분석 함수를 적용한다.
df["상품명_토큰"] = df["상품명"].apply(extract_title_tokens)

# 전처리 전 제목과 전처리 후 토큰을 비교한다.
display(
    df[["상품명", "상품명_토큰"]].head(10)
)

,상품명,상품명_토큰
0,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항
3,테오,테오
4,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자
6,모순,모순
7,서리꽃 세트,서리 꽃 세트
8,데미안,데미안
9,브람스를 좋아하세요,브람스


## 실습 3. 한국어 형태소 분석 적용 결과

### 적용 기준

Kiwi 형태소 분석기를 사용하여 상품명에서 다음 품사만 추출했다.

- `NNG`: 일반 명사
- `NNP`: 고유 명사
- `SL`: 영문 단어

### 전처리 결과

- `한국사 이상현상 연구원(일반판)` → `한국사 이상 현상 연구원 일반판`
- `녹색 절벽의 신자들` → `녹색 절벽 신자`
- `서리꽃 세트` → `서리 꽃 세트`
- `브람스를 좋아하세요` → `브람스`

조사, 어미, 동사 등을 제외하고 분야 구분에 사용할 주요 명사와 고유 명사를 남겼다.

### 확인한 점

형태소 분석을 사용하면 제목에서 핵심 단어만 추출할 수 있다. 다만 `빵충`이 `빵 충`으로 분리되는 것처럼 형태소 분석 결과가 항상 원래 단어의 의미와 정확히 일치하는 것은 아니다.

이후에는 `상품명_토큰` 컬럼을 TF-IDF 입력으로 사용하여 기본 상품명을 사용했을 때와 모델 성능이 달라지는지 확인한다.

### 실습 4. 불필요한 단어 필터링하기

형태소 분석 결과에 있는 모든 토큰이 분류에 유용한 것은 아닙니다.

이번에는 최소한의 필터만 적용합니다.

In [12]:
# 분야 구분에 도움이 적은 불용어를 정의한다.
stopwords = {
    "에디션",
}


# 형태소 분석된 상품명에서 불필요한 토큰을 제거하는 함수를 만든다.
def clean_title_tokens(token_text):
    cleaned_tokens = []

    # 공백을 기준으로 토큰을 하나씩 확인한다.
    for token in str(token_text).split():

        # 길이가 1글자인 토큰은 제외한다.
        if len(token) < 2:
            continue

        # 숫자로만 이루어진 토큰은 제외한다.
        if token.isdigit():
            continue

        # 불용어에 포함된 토큰은 제외한다.
        if token in stopwords:
            continue

        # 조건을 통과한 토큰만 저장한다.
        cleaned_tokens.append(token)

    # 정제된 토큰을 공백으로 연결해 반환한다.
    return " ".join(cleaned_tokens)


# 모든 상품명 토큰에 정제 함수를 적용한다.
df["상품명_정제"] = df["상품명_토큰"].apply(clean_title_tokens)

# 원본 상품명, 형태소 분석 결과, 최종 정제 결과를 비교한다.
display(
    df[["상품명", "상품명_토큰", "상품명_정제"]].head(10)
)

,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스


## 실습 4. 불필요한 단어 필터링 결과

### 적용한 필터

- 길이가 1글자인 토큰 제거
- 숫자로만 이루어진 토큰 제거
- 불용어 `에디션` 제거

### 전처리 결과

- `빵 충 사육 준수 사항` → `사육 준수 사항`
- `서리 꽃 세트` → `서리 세트`
- `한국사 이상 현상 연구원 일반판`은 제거 조건에 해당하는 단어가 없어 그대로 유지되었다.
- `싯다르타`, `테오`, `수족관`처럼 두 글자 이상인 단어도 그대로 유지되었다.

### 확인한 점

불필요한 짧은 토큰과 숫자를 제거해 TF-IDF에 들어갈 단어를 줄였다.

다만 `빵`, `충`, `꽃`처럼 의미가 있을 수 있는 한 글자 단어도 함께 제거되었다. 따라서 전처리를 많이 적용한다고 항상 분류 성능이 좋아지는 것은 아니며, 실제 모델 평가 결과를 통해 효과를 확인해야 한다.

다음 실습에서는 `상품명_정제` 컬럼을 사용해 모델을 다시 학습하고 Baseline 결과와 비교한다.

### 실습 5. Improved 분류 모델 평가하기

이제 형태소 분석과 단어 필터링을 적용한 텍스트로 다시 TF-IDF와 Naive Bayes를 학습합니다.

알고리즘 자체는 바꾸지 않습니다.

In [13]:
# Baseline과 동일한 행을 사용해 정제된 훈련·테스트 제목을 가져온다.
train_texts = df.loc[X_train.index, "상품명_정제"]
test_texts = df.loc[X_test.index, "상품명_정제"]

# 정제된 상품명을 변환할 새로운 TF-IDF 변환기를 만든다.
improved_vectorizer = TfidfVectorizer()

# 훈련 데이터에서 단어를 학습하고 숫자 벡터로 변환한다.
X_train_improved = improved_vectorizer.fit_transform(train_texts)

# 훈련 데이터에서 배운 기준으로 테스트 데이터를 변환한다.
X_test_improved = improved_vectorizer.transform(test_texts)

# Baseline과 동일한 Naive Bayes 모델을 만든다.
improved_model = MultinomialNB()

# 정제된 훈련 데이터로 모델을 학습한다.
improved_model.fit(X_train_improved, y_train)

# 정제된 테스트 데이터의 분야를 예측한다.
improved_pred = improved_model.predict(X_test_improved)

# 실제 분야와 예측 분야를 비교해 정확도를 계산한다.
improved_accuracy = accuracy_score(
    y_test,
    improved_pred
)

# 개선 모델의 전체 정확도와 분야별 성능을 출력한다.
print("Improved Accuracy:", improved_accuracy)

print(
    classification_report(
        y_test,
        improved_pred,
        zero_division=0
    )
)

Improved Accuracy: 0.5788546255506608
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135



## 실습 5. Improved 분류 모델 평가 결과

### 비교 조건

Baseline 모델과 Improved 모델은 동일한 훈련 데이터와 테스트 데이터를 사용했다.

- Baseline: 원본 상품명 → TF-IDF → Naive Bayes
- Improved: 형태소 분석 및 단어 필터링 → TF-IDF → Naive Bayes

모델 알고리즘은 변경하지 않고 TF-IDF에 입력되는 텍스트만 변경했다.

### Before / After

| 모델 | Accuracy | Macro F1 | 정답 수 |
| --- | ---: | ---: | ---: |
| Baseline | 53.74% | 0.39 | 610 / 1,135 |
| Improved | 57.89% | 0.49 | 657 / 1,135 |

### 개선 결과

- 정확도가 약 **4.15%p 상승**했다.
- 정답 수가 **47권 증가**했다.
- Macro F1-score가 **0.39에서 0.49로 상승**했다.
- Baseline에서 예측하지 못했던 `가정/육아` 분야의 F1-score가 **0.00에서 0.36으로 상승**했다.
- `컴퓨터/IT`는 F1-score가 **0.79에서 0.81로 상승**했다.
- `경제/경영`은 F1-score가 **0.60에서 0.72로 상승**했다.
- `인문`은 F1-score가 **0.51에서 0.58로 상승**했다.
- `시/에세이`는 F1-score가 **0.57에서 0.48로 하락**했다.

### 해석

형태소 분석과 단어 필터링을 적용한 결과 전체 정확도와 Macro F1-score가 모두 상승했다.

특히 데이터가 적은 일부 분야의 예측 성능이 개선되었다. 그러나 모든 분야의 성능이 함께 상승한 것은 아니며, `시/에세이`처럼 성능이 낮아진 분야도 있었다.

따라서 이번 전처리는 전체 모델 성능에는 도움이 되었지만, 분야별 결과를 함께 확인하며 개선 효과를 판단해야 한다.

### 실습 6. 추천 결과의 문제 개선하기

이번에는 Chapter 05와 Chapter 06의 추천 기능을 다시 봅니다.

In [14]:
# 도서 사이의 코사인 유사도를 계산하기 위해 불러온다.
from sklearn.metrics.pairwise import cosine_similarity

# 전체 정제 상품명을 추천용 TF-IDF 벡터로 변환한다.
recommend_vectorizer = TfidfVectorizer()

book_vectors = recommend_vectorizer.fit_transform(
    df["상품명_정제"]
)


# 선택한 도서와 비슷한 도서를 추천하는 함수를 만든다.
def recommend_books(selected_index, top_n=5):
    # 선택한 도서의 제목과 분야를 가져온다.
    selected_title = df.loc[selected_index, "상품명"]
    selected_category = df.loc[selected_index, "분야"]

    # 선택한 도서와 같은 분야의 도서만 후보로 가져온다.
    candidate_df = df[
        df["분야"] == selected_category
    ].copy()

    # 선택한 도서 자기 자신은 후보에서 제외한다.
    candidate_df = candidate_df[
        candidate_df.index != selected_index
    ]

    # 같은 분야 후보의 원래 행 번호를 가져온다.
    candidate_indices = candidate_df.index

    # 선택 도서와 같은 분야 후보들의 코사인 유사도를 계산한다.
    similarities = cosine_similarity(
        book_vectors[selected_index],
        book_vectors[candidate_indices]
    ).flatten()

    # 계산한 유사도를 후보 데이터에 추가한다.
    candidate_df["similarity"] = similarities

    # 유사도가 0보다 큰 도서만 남긴다.
    recommendations = candidate_df[
        candidate_df["similarity"] > 0
    ]

    # 유사도가 높은 순서로 정렬하고 최대 5권만 선택한다.
    recommendations = recommendations.sort_values(
        by="similarity",
        ascending=False
    ).head(top_n)

    print("선택 도서:", selected_title)
    print("선택 분야:", selected_category)

    # 추천 결과가 없으면 안내 문구를 출력한다.
    if recommendations.empty:
        print("현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.")
        return recommendations

    # 추천 결과가 있으면 필요한 컬럼만 보여준다.
    return recommendations[
        ["상품명", "인물", "출판사", "분야", "similarity"]
    ]


# 현재 데이터에 존재하는 대표 도서를 선택한다.
selected_title = "소년이 온다"

# 선택한 제목의 행 번호를 찾는다.
selected_index = df.index[
    df["상품명"] == selected_title
][0]

# 비슷한 도서를 최대 5권 추천한다.
recommend_books(selected_index)

선택 도서: 소년이 온다
선택 분야: 소설


,상품명,인물,출판사,분야,similarity
696,소년이로,편혜영,문학과지성사,소설,1.000000
320,바다에서 온 소년,개럿 카,북파머스,소설,0.722145


## 실습 6. 추천 결과 개선

### 적용한 개선 조건

- 선택한 도서와 같은 분야만 추천 후보로 사용했다.
- 선택한 도서 자기 자신은 추천에서 제외했다.
- 형태소 분석과 단어 필터링을 적용한 제목을 사용했다.
- 코사인 유사도가 0보다 큰 도서만 남겼다.
- 추천 결과가 5권보다 적더라도 유사도 0인 도서로 채우지 않았다.

### 실행 결과

선택 도서로 `소년이 온다`를 사용했다.

- 선택 분야: 소설
- 양수 유사도를 가진 추천 도서: 2권
- `소년이로`: 유사도 1.0000
- `바다에서 온 소년`: 유사도 0.7221

### 해석

`소년이 온다`와 제목에서 공통 특징을 가진 소설은 2권만 확인되었다.

기존처럼 무조건 5권을 출력했다면 유사도가 0인 도서도 추천 결과에 포함될 수 있다. 개선된 추천 로직은 유사도가 0보다 큰 도서만 보여주기 때문에 추천 결과가 2권만 출력되었다.

추천 개수를 항상 5권으로 맞추는 것보다 공통 특징이 없는 도서를 제외하여 근거 없는 추천을 줄이는 것이 중요하다.

### 실습 7. Streamlit 앱에서 최종 검증하기

## 실습 7 결과

- Streamlit 앱에서 `books_improved.csv`를 정상적으로 불러왔다.
- Kiwi 형태소 분석과 불용어 제거를 적용한 제목으로 도서 분야를 예측했다.
- 같은 분야에서 코사인 유사도가 0보다 큰 도서만 추천하도록 구현했다.
- 선택한 도서 자기 자신은 추천 결과에서 제외했다.
- `바보빅터`와 `바보 빅터`처럼 띄어쓰기가 달라지면 예측 결과도 달라질 수 있음을 확인했다.

## 최종 결론

기존 모델의 정확도는 약 **53.7%**였으며, 형태소 분석과 불용어 제거를 적용한 개선 모델의 정확도는 약 **57.9%**였다.

전처리를 통해 정확도가 약 **4.2%p 향상**되었지만, 제목만으로 9개 분야를 분류하는 데에는 한계가 있었다. 특히 띄어쓰기 차이와 짧은 제목에 민감했고, 데이터가 적은 분야의 예측 성능도 상대적으로 낮았다.

앞으로 카테고리별 데이터 수를 균형 있게 구성하고 저자, 출판사 등의 정보를 함께 사용하면 분류 성능을 더 개선할 수 있다.